In [0]:

from pyspark.sql.functions import (
    col, 
    date_format, 
    current_timestamp
)
from pyspark.sql.types import DecimalType

CATALOG_NAME = "banking_lakehouse_db2"
SCHEMA_NAME = "gold"
TABLE_NAME = "transaction_fraud_analytics"
FULL_TABLE_NAME = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{TABLE_NAME}"
GOLD_PATH = f"abfss://gold@bankingdelakevishal.dfs.core.windows.net/{TABLE_NAME}/"



In [0]:
# ============================================
# GOLD: TRANSACTION FRAUD ANALYTICS
# ============================================
# 1. Read Silver Tables
fact_tx = spark.table(f"`{CATALOG_NAME}`.silver.transaction")
dim_account = spark.table(f"`{CATALOG_NAME}`.silver.account")
dim_customer = spark.table(f"`{CATALOG_NAME}`.silver.customer")

# 2. Join Contextual Data for Fraud Analytics
fraud_analytics_df = (
    fact_tx
    .join(dim_account.select("account_id", "customer_id", "account_type", "account_status"), "account_id", "inner")
    .join(dim_customer.select("customer_id", "risk_category", "kyc_status", "city", "state", "country"), "customer_id", "inner")
    .select(
        fact_tx["transaction_id"],
        fact_tx["account_id"],
        dim_account["customer_id"],
        fact_tx["transaction_timestamp"],
        date_format(fact_tx["transaction_timestamp"], "yyyy-MM-dd").alias("transaction_date"),
        date_format(fact_tx["transaction_timestamp"], "HH:00").alias("transaction_hour"),
        fact_tx["transaction_type"],
        fact_tx["channel"],
        fact_tx["merchant"],
        fact_tx["amount"].cast(DecimalType(18, 2)).alias("amount"),
        fact_tx["currency"],
        fact_tx["transaction_status"],
        fact_tx["fraud_flag"],
        dim_account["account_type"],
        dim_account["account_status"],
        dim_customer["risk_category"].alias("customer_risk_category"),
        dim_customer["kyc_status"],
        dim_customer["city"].alias("customer_city"),
        dim_customer["state"].alias("customer_state"),
        dim_customer["country"].alias("customer_country"),
        current_timestamp().alias("gold_ingestion_timestamp")
    )
)

# 3. Write to ADLS & Unity Catalog
fraud_analytics_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PATH)
fraud_analytics_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FULL_TABLE_NAME)

print(f"Transaction Fraud Analytics Gold Table Created: {fraud_analytics_df.count()} records written to '{FULL_TABLE_NAME}'.")

Transaction Fraud Analytics Gold Table Created: 96815 records written to '`banking_lakehouse_db2`.gold.transaction_fraud_analytics'.
